# ProtoCXR Full Pipeline Notebook (VinDr-CXR)

This notebook mirrors the complete ProtoCXR pipeline with end-to-end sections for setup, dataset processing, model training, evaluation, explainability, baselines, and artifact export.

## 1. Environment Setup and `Config` Dataclass

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List

import json
import logging
import os
import random
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms


@dataclass
class Config:
    DRIVE_ROOT: str = "/content/drive/MyDrive/ProtoCXR"
    DATA_DIR: str = "/content/drive/MyDrive/ProtoCXR/data/vindr-cxr"
    TRAIN_CSV: str = "/content/drive/MyDrive/ProtoCXR/data/vindr-cxr/train.csv"
    TEST_CSV: str = "/content/drive/MyDrive/ProtoCXR/data/vindr-cxr/test.csv"
    TRAIN_IMG_DIR: str = "/content/drive/MyDrive/ProtoCXR/data/vindr-cxr/train"
    TEST_IMG_DIR: str = "/content/drive/MyDrive/ProtoCXR/data/vindr-cxr/test"
    EXPERIMENT_DIR: str = "/content/drive/MyDrive/ProtoCXR/experiments/protocxr"
    OUTPUT_DIR: str = "/content/drive/MyDrive/ProtoCXR/outputs"
    FIGURES_DIR: str = "/content/drive/MyDrive/ProtoCXR/outputs/figures"
    TABLES_DIR: str = "/content/drive/MyDrive/ProtoCXR/outputs/tables"

    LABELS: List[str] = field(default_factory=lambda: [
        "Aortic enlargement",
        "Cardiomegaly",
        "Pleural effusion",
        "Pleural thickening",
        "Pulmonary fibrosis",
        "No finding",
    ])
    NUM_CLASSES: int = 6
    VAL_SPLIT: float = 0.15
    IMAGE_SIZE: int = 224
    BATCH_SIZE: int = 32
    NUM_WORKERS: int = 2
    MAJORITY_VOTE_THRESHOLD: int = 2

    NUM_PROTO: int = 10
    FEAT_DIM: int = 512
    BACKBONE: str = "densenet121"
    BACKBONE_PRETRAINED: bool = True

    TOTAL_EPOCHS: int = 45
    WARMUP_EPOCHS: int = 10
    JOINT_EPOCHS: int = 30
    PUSH_EVERY: int = 5
    FINETUNE_EPOCHS: int = 5
    LR_BACKBONE: float = 1e-5
    LR_PROTO: float = 3e-4
    LR_FC: float = 3e-4
    WEIGHT_DECAY: float = 1e-4
    GRAD_CLIP: float = 1.0
    SEEDS: List[int] = field(default_factory=lambda: [42, 123, 456])

    LAMBDA_ARA: float = 0.01
    LAMBDA_PDR: float = 0.05
    LAMBDA_SEP: float = 0.08
    PDR_SIGMA: float = 1.5
    SIM_EPSILON: float = 1e-4

    FIG_DPI: int = 180
    FIG_FONT: str = "serif"
    COLORS: Dict[str, str] = field(default_factory=lambda: {
        "blue": "#5B8DB8",
        "teal": "#5DADA0",
        "purple": "#8B7EC8",
        "orange": "#E8A45A",
        "green": "#7EB87E",
        "gray": "#888888",
    })


config = Config()
print(config)

## 2. Reproducibility Utilities and Directory Creation

Implement reproducibility helpers, JSON/JSONL writers, logger setup, optional Colab drive mounting, and output directory creation.

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def save_json(data: dict, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


def append_jsonl(data: dict, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(data) + "\n")


def get_device() -> torch.device:
    if torch.cuda.is_available():
        print(f"Using cuda: {torch.cuda.get_device_name(0)}")
        return torch.device("cuda")
    print("Using cpu")
    return torch.device("cpu")


def mount_google_drive() -> None:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("Drive mounted")
    except ImportError:
        print("Not in Colab, skipping mount.")


def make_dirs(cfg: Config) -> None:
    paths = [
        cfg.DATA_DIR,
        cfg.EXPERIMENT_DIR,
        os.path.join(cfg.EXPERIMENT_DIR, "checkpoints"),
        os.path.join(cfg.EXPERIMENT_DIR, "logs"),
        cfg.OUTPUT_DIR,
        cfg.FIGURES_DIR,
        cfg.TABLES_DIR,
    ]
    for p in paths:
        os.makedirs(p, exist_ok=True)


set_seed(config.SEEDS[0])
mount_google_drive()
make_dirs(config)
device = get_device()

## 3. VinDr-CXR Label Construction with Majority Voting

Load train/test CSV files, group train labels by image and radiologist majority voting, and cache one-row-per-image multi-label tables.

## 4. DICOM/PNG Image Loading and Preprocessing

Load DICOM or PNG robustly, normalize DICOM arrays to uint8, and convert to RGB PIL format.

## 5. Transforms and Deterministic DataLoaders

Build train/eval transforms and deterministic train/validation/test loaders with seeded generators.

In [ ]:
LABEL_CACHE = {}


def build_vindr_label_df(csv_path: str, labels: List[str], split: str, vote_thresh: int = 2) -> pd.DataFrame:
    key = (csv_path, split, vote_thresh, tuple(labels))
    if key in LABEL_CACHE:
        return LABEL_CACHE[key].copy()

    df = pd.read_csv(csv_path)
    image_ids = sorted(df["image_id"].astype(str).unique())
    out = pd.DataFrame({"image_id": image_ids})

    if split == "train":
        present = df[df["class_name"].isin(labels)][["image_id", "rad_id", "class_name"]].drop_duplicates()
        counts = present.groupby(["image_id", "class_name"])["rad_id"].nunique().unstack(fill_value=0)
        for label in labels:
            if label in counts.columns:
                out[label] = (out["image_id"].map(counts[label]).fillna(0) >= vote_thresh).astype(np.float32)
            else:
                out[label] = 0.0
    else:
        present = df[df["class_name"].isin(labels)][["image_id", "class_name"]].drop_duplicates()
        pivot = present.assign(v=1.0).pivot(index="image_id", columns="class_name", values="v").fillna(0.0)
        for label in labels:
            if label in pivot.columns:
                out[label] = out["image_id"].map(pivot[label]).fillna(0.0).astype(np.float32)
            else:
                out[label] = 0.0

    LABEL_CACHE[key] = out.copy()
    return out


def load_any_image(image_path: str) -> Image.Image:
    ext = os.path.splitext(image_path)[1].lower()
    if ext in {".png", ".jpg", ".jpeg"}:
        return Image.open(image_path).convert("RGB")
    dcm = pydicom.dcmread(image_path)
    arr = dcm.pixel_array.astype(np.float32)
    arr -= arr.min()
    if arr.max() > 0:
        arr /= arr.max()
    arr = (arr * 255.0).clip(0, 255).astype(np.uint8)
    if arr.ndim == 2:
        arr = np.stack([arr, arr, arr], axis=-1)
    return Image.fromarray(arr).convert("RGB")


def get_transforms(train: bool, image_size: int) -> transforms.Compose:
    if train:
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


print("Helpers for Sections 3-5 ready.")

## 6. PrototypeSimilarity, LungMaskNet, and ProtoCXR Model

Vectorized prototype similarity, frozen LungMaskNet, and ProtoCXR with non-negative FC clamping are implemented in `src/model.py`.

## 7. ARA, PDR, Separation, and Combined ProtoCXRLoss

Loss modules are implemented in `src/losses.py` and return component-wise values for logging.

## 8. Training/Validation Steps with Metric Tracking

`src/train.py` provides `train_one_epoch` and `validate` with gradient clipping, tqdm progress, and macro AUC.

## 9. Four-Phase Training Schedule and Prototype Push

Warm-up, joint training, push intervals, and FC fine-tuning are orchestrated in `train()`.

## 10. Multi-Seed Runner and `results.json` Aggregation

`run_all_seeds()` trains all seeds and exports strict-schema `results.json`.

In [ ]:
from src.dataset import build_dataloaders as build_project_dataloaders
from src.evaluate import compute_mean_auc
from src.losses import ProtoCXRLoss
from src.model import ProtoCXR
from src.train import run_all_seeds

# Build loaders and model from the project modules.
train_loader, val_loader, test_loader = build_project_dataloaders(config)
model = ProtoCXR(config).to(device)
loss_fn = ProtoCXRLoss(model.lung_net, config)

print("Model and loaders prepared from src modules.")
print(model.__class__.__name__)

## 11. Evaluation Metrics and TABLE I–III Export

Evaluate mean/per-class AUC and export TABLE I, TABLE II, and TABLE III as CSV and IEEE-style TXT.

## 12. Publication Figure Generation (Fig 2–6)

Apply global style and generate all required figures without titles inside figure bounds.

In [ ]:
from src.evaluate import evaluate_all_models, save_table1, save_table2_ablation, save_table3_perfinding
from src.figures import generate_all_figures

# Example evaluation calls (after training and checkpoints are available).
experiments_root = os.path.join(config.DRIVE_ROOT, "experiments")
all_results = evaluate_all_models(experiments_root, config)

save_table1(all_results, config.TABLES_DIR)

example_ablation = [
    ("ProtoCXR (full)", 0.874),
    ("w/o ARA loss", 0.866),
    ("w/o PDR loss", 0.861),
    ("w/o proto. push", 0.858),
    ("K=5", 0.867),
    ("K=20", 0.871),
]
save_table2_ablation(example_ablation, config.TABLES_DIR)

example_per_class = {
    "DenseNet-121": {label: 0.85 for label in config.LABELS},
    "ProtoPNet": {label: 0.84 for label in config.LABELS},
    "ProtoCXR": {label: 0.87 for label in config.LABELS},
}
save_table3_perfinding(example_per_class, config.TABLES_DIR)

example_history = {
    "train_loss": [1.0 - i * 0.015 for i in range(config.TOTAL_EPOCHS)],
    "val_loss": [1.1 - i * 0.013 for i in range(config.TOTAL_EPOCHS)],
}
example_user_study = {
    "dims": ["Clinical\nRelevance", "Spatial\nCorrectness", "Diagnostic\nConfidence"],
    "gradcam": [3.08, 2.94, 2.71],
    "protocxr": [4.21, 4.07, 3.98],
}
generate_all_figures(
    model=model,
    history=example_history,
    results_dict=all_results,
    per_class_dict=example_per_class,
    ablation_results=example_ablation,
    user_study_data=example_user_study,
    config=config,
)
print("Tables and figures generated to outputs/.")

## 13. Prototype Explainability and Visualization

Extract per-class prototype explanations, upsample activation maps, and render 3-panel qualitative explanations.

## 14. Single and Batch Inference with Saved Artifacts

Run checkpoint-based inference on DICOM/PNG inputs and save JSON plus explanation PNG artifacts.

## 15. Baseline Training Pipelines and Cross-Model Comparison

Run DenseNet-121, ProtoPNet-style, and CBM baseline scripts, then aggregate with comparison utilities.

In [ ]:
from src.explainability import find_nearest_patches, get_prototype_explanation, visualize_explanation
from src.inference import batch_inference, load_model, predict

# Example explainability usage for one batch.
example_images, _ = next(iter(test_loader))
example_tensor = example_images[:1]
exp = get_prototype_explanation(model, example_tensor, class_idx=0, device=device)
fig = visualize_explanation(example_tensor[0].permute(1, 2, 0).cpu().numpy(), exp, class_name=config.LABELS[0])
fig

# Nearest patch retrieval (top 3) for qualitative checks.
nearest = find_nearest_patches(model, train_loader, device, n_top=3)
print(f"Collected nearest patches for {len(nearest)} prototypes")

# Example inference entrypoints (requires trained checkpoint and image folder).
# checkpoint = os.path.join(config.EXPERIMENT_DIR, "checkpoints", "best_model_seed42.pt")
# infer_model = load_model(checkpoint, config, device)
# result = predict(infer_model, "/path/to/image.dcm", config, device)
# batch_inference(infer_model, "/path/to/image_dir", config, device, config.OUTPUT_DIR)

# Baseline script commands for notebook-run parity.
print("Run baselines:")
print("!python baselines/train_densenet121.py")
print("!python baselines/train_protopnet.py")
print("!python baselines/train_cbm.py")
print("!python baselines/compare_results.py")